In [45]:
import pickle
import sys
from pathlib import Path
import numpy as np
sys.path.append("..")



In [46]:
# ============================================================
# CONFIGURATION - Update these paths to your data
# ============================================================

DATA_DIR = Path('../pickle_files/data_28-01-2026_19-27-13')

MEASUREMENTS_FILE = DATA_DIR / 'measurements.pkl'
GRAPH_FILE = DATA_DIR / 'graph.pkl'
GROUND_TRUTH_FILE = DATA_DIR / 'ground_truth.pkl'

# Verify files exist
print("Checking input files:")
for f in [MEASUREMENTS_FILE, GRAPH_FILE, GROUND_TRUTH_FILE]:
    status = "✓" if f.exists() else "✗"
    print(f"  {status} {f.name}: {f.exists()}")

Checking input files:
  ✓ measurements.pkl: True
  ✓ graph.pkl: True
  ✓ ground_truth.pkl: True


In [47]:
class TimeAwareMeasurement:
    """Data class for syndrome measurements with timing information."""
    def __init__(self, path_edges, histogram, duration, latency_stats=None, code_type='513'):
        """
        :param path_edges: List of tuples representing the path, e.g., [(0,1), (1,2)]
        :param histogram: A numpy array of syndrome counts.
        :param duration: The duration or average latency of this measurement (seconds).
        :param latency_stats: A dictionary with 'max', 'std', 'count'.
        :param code_type: String indicating the code type, e.g., '313_X', '313_Z', '513'.
        """
        self.path_edges = path_edges
        self.histogram = histogram
        self.duration = duration
        self.latency_stats = latency_stats
        self.code_type = code_type

In [48]:
# ============================================================
# LOAD PICKLE FILES
# ============================================================

with open(MEASUREMENTS_FILE, 'rb') as f:
    measurements = pickle.load(f)

with open(GRAPH_FILE, 'rb') as f:
    graph = pickle.load(f)

with open(GROUND_TRUTH_FILE, 'rb') as f:
    ground_truth = pickle.load(f)

print(f"Loaded {len(measurements)} syndrome measurements")
print(f"Loaded graph with {len(graph.nodes)} nodes and {len(graph.edges)} edges")
print(f"Loaded ground truth for {len(ground_truth)} edges")

Loaded 40 syndrome measurements
Loaded graph with 6 nodes and 7 edges
Loaded ground truth for 7 edges


In [49]:
print(ground_truth)

{(0, 1): {'p_bit_flip': 0.04541015625, 'p_phase_flip': 0.0478515625, 'z_counts': {'1': 186, '0': 3910}, 'x_counts': {'1': 196, '0': 3900}}, (0, 3): {'p_bit_flip': 0.038818359375, 'p_phase_flip': 0.05078125, 'z_counts': {'1': 159, '0': 3937}, 'x_counts': {'1': 208, '0': 3888}}, (1, 2): {'p_bit_flip': 0.0458984375, 'p_phase_flip': 0.06103515625, 'z_counts': {'1': 188, '0': 3908}, 'x_counts': {'1': 250, '0': 3846}}, (1, 4): {'p_bit_flip': 0.515869140625, 'p_phase_flip': 0.492919921875, 'z_counts': {'1': 2113, '0': 1983}, 'x_counts': {'1': 2019, '0': 2077}}, (2, 5): {'p_bit_flip': 0.03125, 'p_phase_flip': 0.037109375, 'z_counts': {'1': 128, '0': 3968}, 'x_counts': {'1': 152, '0': 3944}}, (3, 4): {'p_bit_flip': 0.068603515625, 'p_phase_flip': 0.07470703125, 'z_counts': {'1': 281, '0': 3815}, 'x_counts': {'1': 306, '0': 3790}}, (4, 5): {'p_bit_flip': 0.041015625, 'p_phase_flip': 0.054443359375, 'z_counts': {'1': 168, '0': 3928}, 'x_counts': {'1': 223, '0': 3873}}}


In [50]:
for measure in measurements:
    print(measure.path_edges, measure.code_type, measure.histogram / np.sum(measure.histogram), measure.duration)

[(0, 1), (1, 2)] 313_X [0.751709   0.08374023 0.09594727 0.06860352] 0.0
[(0, 1), (1, 2)] 313_Z [0.7121582  0.08789062 0.11572266 0.08422852] 0.0
[(0, 3), (3, 4)] 313_X [0.6687012  0.11010742 0.11621094 0.10498047] 0.0
[(0, 3), (3, 4)] 313_Z [0.6555176  0.11450195 0.12158203 0.10839844] 0.0
[(0, 1), (1, 4)] 313_X [0.25073242 0.24682617 0.25463867 0.24780273] 0.0
[(0, 1), (1, 4)] 313_Z [0.25097656 0.26342773 0.22583008 0.25976562] 0.0
[(1, 4), (4, 3)] 313_X [0.25170898 0.24609375 0.25561523 0.24658203] 0.0
[(1, 4), (4, 3)] 313_Z [0.24951172 0.23999023 0.25756836 0.2529297 ] 0.0
[(1, 0), (0, 3)] 313_X [0.7216797  0.08984375 0.10620117 0.08227539] 0.0
[(1, 0), (0, 3)] 313_Z [0.6994629  0.09619141 0.11254883 0.09179688] 0.0
[(1, 4), (4, 5)] 313_X [0.24365234 0.24584961 0.25268555 0.2578125 ] 0.0
[(1, 4), (4, 5)] 313_Z [0.2541504  0.24121094 0.25       0.25463867] 0.0
[(1, 2), (2, 5)] 313_X [0.7775879  0.07592773 0.08105469 0.06542969] 0.0
[(1, 2), (2, 5)] 313_Z [0.74072266 0.09130859 0.087

In [51]:
bit_flip_syndromes = [m for m in measurements if m.code_type == '313_X']
phase_flip_syndromes = [m for m in measurements if m.code_type == '313_Z']

In [52]:
import pandas as pd

print("Bit Syndrome Analysis")
print("=" * 80)

error_key = 'p_bit_flip'
rows = []
for measure in bit_flip_syndromes:
    error_per_edge = [ground_truth.get(tuple(sorted(edge))).get(error_key) for edge in measure.path_edges]
    edge_no_error = [1-e for e in error_per_edge]
    p_no_error = np.prod(edge_no_error)
    syndrome_no_error = measure.histogram[0]/np.sum(measure.histogram)
    
    rows.append({
        'Path': str(measure.path_edges),
        'Error/Edge': str([round(e, 4) for e in error_per_edge]),
        'P(no error)': round(p_no_error, 4),
        'Syndrome (no err)': round(syndrome_no_error, 4)
    })

df = pd.DataFrame(rows)
display(df)

Bit Syndrome Analysis


,Path,Error/Edge,P(no error),Syndrome (no err)
0,"[(0, 1), (1, 2)]","[0.0454, 0.0459]",0.9108,0.7517
1,"[(0, 3), (3, 4)]","[0.0388, 0.0686]",0.8952,0.6687
2,"[(0, 1), (1, 4)]","[0.0454, 0.5159]",0.4621,0.2507
3,"[(1, 4), (4, 3)]","[0.5159, 0.0686]",0.4509,0.2517
4,"[(1, 0), (0, 3)]","[0.0454, 0.0388]",0.9175,0.7217
5,"[(1, 4), (4, 5)]","[0.5159, 0.041]",0.4643,0.2437
6,"[(1, 2), (2, 5)]","[0.0459, 0.0312]",0.9243,0.7776
7,"[(2, 1), (1, 0)]","[0.0459, 0.0454]",0.9108,0.7524
8,"[(2, 5), (5, 4)]","[0.0312, 0.041]",0.9290,0.7188
9,"[(2, 1), (1, 4)]","[0.0459, 0.5159]",0.4619,0.2422


In [53]:
print("Phase Syndrome Analysis")
print("=" * 80)

error_key = 'p_phase_flip'
rows = []
for measure in phase_flip_syndromes:
    error_per_edge = [ground_truth.get(tuple(sorted(edge))).get(error_key) for edge in measure.path_edges]
    edge_no_error = [1-e for e in error_per_edge]
    p_no_error = np.prod(edge_no_error)
    syndrome_no_error = measure.histogram[0]/np.sum(measure.histogram)
    
    rows.append({
        'Path': str(measure.path_edges),
        'Error/Edge': str([round(e, 4) for e in error_per_edge]),
        'P(no error)': round(p_no_error, 4),
        'Syndrome (no err)': round(syndrome_no_error, 4)
    })

df = pd.DataFrame(rows)
display(df)

Phase Syndrome Analysis


,Path,Error/Edge,P(no error),Syndrome (no err)
0,"[(0, 1), (1, 2)]","[0.0479, 0.061]",0.8940,0.7122
1,"[(0, 3), (3, 4)]","[0.0508, 0.0747]",0.8783,0.6555
2,"[(0, 1), (1, 4)]","[0.0479, 0.4929]",0.4828,0.2510
3,"[(1, 4), (4, 3)]","[0.4929, 0.0747]",0.4692,0.2495
4,"[(1, 0), (0, 3)]","[0.0479, 0.0508]",0.9038,0.6995
5,"[(1, 4), (4, 5)]","[0.4929, 0.0544]",0.4795,0.2542
6,"[(1, 2), (2, 5)]","[0.061, 0.0371]",0.9041,0.7407
7,"[(2, 1), (1, 0)]","[0.061, 0.0479]",0.8940,0.7153
8,"[(2, 5), (5, 4)]","[0.0371, 0.0544]",0.9105,0.6733
9,"[(2, 1), (1, 4)]","[0.061, 0.4929]",0.4761,0.2527
